# Example Application of Results of NorCOM to a population DVector

In [1]:
from pathlib import Path
import math
from functools import reduce

import pandas as pd
from caf.base import DVector

### Define paths to the NorCOM estimation results and the DVector we want to apply the parameters to

In [2]:
# path to results of the parameter estimation
# Note this may be changed to work with the pickle files instead of the 
results_path = Path(r'I:\NorMITs NorCOM\AECOM working\v15')

In [3]:
# path to a household DVector to apply the NorCOM estimation results to
hh_data_path = Path(r"F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P4.3_NE.hdf")

and read in the dvector data

In [4]:
hh_data = DVector.load(hh_data_path)

### Get the segmentation and zone system from the hh data

In [5]:
dvec_segmentation = hh_data.segmentation

In [6]:
dvec_segmentation.names

['children', 'accom_h', 'ns_sec', 'adults', 'car_availability', 'adult_nssec']

In [7]:
zone_system = hh_data.zoning_system

In [8]:
zone_system.name

'LSOA2021-NE'

### Read in the final estimated model for the two car ownership submodels


In [9]:
model_dirs = ['0v1', '1v2+']
results = {}
variables = []
for model in model_dirs:
    coefficients = pd.read_csv(results_path / model / 'output' / 'final_model_coefficients.csv')
    results[model] = coefficients[['Feature', 'Coefficient']]
    variables += list(coefficients['Feature'].apply(lambda x: '_'.join(x.split('_')[:-1])))
variables = list(set(variables))
variables = ['intercept' if x=='' else x for x in variables]

### Get the relevant segmentation in caf.base that represents the variables used in the model

In [10]:
variables

['intercept',
 'hh_adults_seg',
 'descta_b01id',
 'tfn_at_v2',
 'walkrail_b01id',
 'hh_children_seg',
 'ns',
 'hh_income_banded_v3']

In [11]:
# TODO this should be somehow automated or specified within the constants, this is just for testing
_MAPPING = {
    'ns_sec': 'ns',
    'children': 'hh_children_seg',
    'adults': 'hh_adults_seg'
}

### Aggregate the input dvector to the segmentation required by the estimation results

In [12]:
hh_data_aggregated = hh_data.aggregate(list(_MAPPING.keys()))

In [13]:
# take a copy of the data to generate the parameter interpretations
norcom_dvec = hh_data_aggregated * 0

In [14]:
# get the values of each of these segmentations in the data
hh_data_aggregated.segmentation.seg_dict

{'children': Segment(name='children', values={1: 'Household with no children or all children non-dependent', 2: 'Household with one or more dependent children'}, alias=None, exclusions=[Exclusion(other_name='age_11', exclusions={1: {1, 2, 3}}), Exclusion(other_name='age_9', exclusions={1: {1, 2, 3}}), Exclusion(other_name='age_ntem', exclusions={1: {1}}), Exclusion(other_name='age_5', exclusions={1: {1}}), Exclusion(other_name='economic_status', exclusions={1: {-8}})], lookups=[]),
 'ns_sec': Segment(name='ns_sec', values={1: 'HRP managerial / professional', 2: 'HRP intermediate / technical', 3: 'HRP semi-routine / routine', 4: 'HRP never worked / long-term unemployed', 5: 'HRP no category, inc. full-time student'}, alias=None, exclusions=[], lookups=[]),
 'adults': Segment(name='adults', values={1: 'No adults or 1 adult in household', 2: '2 adults in household', 3: '3 or more adults in household'}, alias=None, exclusions=[], lookups=[])}

In [15]:
# get model instance and the intercept of the model
model = '0v1'
results_data = results.get(model)
intercept = results_data[results_data['Feature'] == 'intercept']['Coefficient'].sum()
# add the intercept to the norcom_dvec
# TODO check the application of the intercept, seems to skew probabilities but this might be because its just an example
norcom_dvec.data = norcom_dvec.data + intercept

In [16]:
# create list of output dvectors to sum
output = [norcom_dvec]

# loop through all the segments
for segment_name in hh_data_aggregated.segmentation.seg_dict.keys():
    segment_values = list(hh_data_aggregated.segmentation.seg_dict.get(segment_name).values.keys())

    # create a segment specific DVector
    segment_dvec = norcom_dvec * 0
    # get the model coefficients that relate to this segment and these values
    for val in segment_values:
        coefficient = results_data[results_data['Feature'] == f'{_MAPPING.get(segment_name)}_{val}']['Coefficient'].sum()
        # replace row values in norcom_dvec to the coefficients
        segment_dvec.data[(segment_dvec.data.index.get_level_values(segment_name) == val)] = coefficient
    output.append(segment_dvec)

In [17]:
# sum the terms of the logit model function across the relevant segments
model_totals = reduce(lambda x, y: x+y, output)
model_totals.data

LSOA2021                E01008162  E01008163  E01008164  E01008165  E01008166  \
ns_sec adults children                                                          
1      1      1          1.971195   1.971195   1.971195   1.971195   1.971195   
              2          2.346954   2.346954   2.346954   2.346954   2.346954   
       2      1          3.107935   3.107935   3.107935   3.107935   3.107935   
              2          3.483694   3.483694   3.483694   3.483694   3.483694   
       3      1          3.191446   3.191446   3.191446   3.191446   3.191446   
              2          3.567205   3.567205   3.567205   3.567205   3.567205   
2      1      1          1.662489   1.662489   1.662489   1.662489   1.662489   
              2          2.038248   2.038248   2.038248   2.038248   2.038248   
       2      1          2.799229   2.799229   2.799229   2.799229   2.799229   
              2          3.174988   3.174988   3.174988   3.174988   3.174988   
       3      1          2.882740   2.882740   2.882740   2.882740   2.882740   
              2          3.258500   3.258500   3.258500   3.258500   3.258500   
3      1      1          0.945572   0.945572   0.945572   0.945572   0.945572   
              2          1.321332   1.321332   1.321332   1.321332   1.321332   
       2      1          2.082313   2.082313   2.082313   2.082313   2.082313   
              2          2.458072   2.458072   2.458072   2.458072   2.458072   
       3      1          2.165824   2.165824   2.165824   2.165824   2.165824   
              2          2.541583   2.541583   2.541583   2.541583   2.541583   
4      1      1          0.149380   0.149380   0.149380   0.149380   0.149380   
              2          0.525139   0.525139   0.525139   0.525139   0.525139   
       2      1          1.286120   1.286120   1.286120   1.286120   1.286120   
              2          1.661879   1.661879   1.661879   1.661879   1.661879   
       3      1          1.369632   1.369632   1.369632   1.369632   1.369632   
              2          1.745391   1.745391   1.745391   1.745391   1.745391   
5      1      1          0.728745   0.728745   0.728745   0.728745   0.728745   
              2          1.104504   1.104504   1.104504   1.104504   1.104504   
       2      1          1.865485   1.865485   1.865485   1.865485   1.865485   
              2          2.241245   2.241245   2.241245   2.241245   2.241245   
       3      1          1.948997   1.948997   1.948997   1.948997   1.948997   
              2          2.324756   2.324756   2.324756   2.324756   2.324756   

LSOA2021                E01008167  E01008170  E01008172  E01008173  E01008174  \
ns_sec adults children                                                          
1      1      1          1.971195   1.971195   1.971195   1.971195   1.971195   
              2          2.346954   2.346954   2.346954   2.346954   2.346954   
       2      1          3.107935   3.107935   3.107935   3.107935   3.107935   
              2          3.483694   3.483694   3.483694   3.483694   3.483694   
       3      1          3.191446   3.191446   3.191446   3.191446   3.191446   
              2          3.567205   3.567205   3.567205   3.567205   3.567205   
2      1      1          1.662489   1.662489   1.662489   1.662489   1.662489   
              2          2.038248   2.038248   2.038248   2.038248   2.038248   
       2      1          2.799229   2.799229   2.799229   2.799229   2.799229   
              2          3.174988   3.174988   3.174988   3.174988   3.174988   
       3      1          2.882740   2.882740   2.882740   2.882740   2.882740   
              2          3.258500   3.258500   3.258500   3.258500   3.258500   
3      1      1          0.945572   0.945572   0.945572   0.945572   0.945572   
              2          1.321332   1.321332   1.321332   1.321332   1.321332   
       2      1          2.082313   2.082313   2.082313   2.082313   2.082313   
              2          2.458

In [18]:
# get DVector of 'e's to apply df.pow to (i.e. we need to do e^-model_totals to get the logit application)
exponential = (model_totals.copy() * 0) + math.e
exponential.data = exponential.data.pow(-model_totals.data)
exponential.data

LSOA2021                E01008162  E01008163  E01008164  E01008165  E01008166  \
ns_sec adults children                                                          
1      1      1          0.139290   0.139290   0.139290   0.139290   0.139290   
              2          0.095660   0.095660   0.095660   0.095660   0.095660   
       2      1          0.044693   0.044693   0.044693   0.044693   0.044693   
              2          0.030694   0.030694   0.030694   0.030694   0.030694   
       3      1          0.041112   0.041112   0.041112   0.041112   0.041112   
              2          0.028235   0.028235   0.028235   0.028235   0.028235   
2      1      1          0.189666   0.189666   0.189666   0.189666   0.189666   
              2          0.130257   0.130257   0.130257   0.130257   0.130257   
       2      1          0.060857   0.060857   0.060857   0.060857   0.060857   
              2          0.041795   0.041795   0.041795   0.041795   0.041795   
       3      1          0.055981   0.055981   0.055981   0.055981   0.055981   
              2          0.038446   0.038446   0.038446   0.038446   0.038446   
3      1      1          0.388457   0.388457   0.388457   0.388457   0.388457   
              2          0.266780   0.266780   0.266780   0.266780   0.266780   
       2      1          0.124642   0.124642   0.124642   0.124642   0.124642   
              2          0.085600   0.085600   0.085600   0.085600   0.085600   
       3      1          0.114655   0.114655   0.114655   0.114655   0.114655   
              2          0.078742   0.078742   0.078742   0.078742   0.078742   
4      1      1          0.861242   0.861242   0.861242   0.861242   0.861242   
              2          0.591473   0.591473   0.591473   0.591473   0.591473   
       2      1          0.276341   0.276341   0.276341   0.276341   0.276341   
              2          0.189782   0.189782   0.189782   0.189782   0.189782   
       3      1          0.254201   0.254201   0.254201   0.254201   0.254201   
              2          0.174577   0.174577   0.174577   0.174577   0.174577   
5      1      1          0.482514   0.482514   0.482514   0.482514   0.482514   
              2          0.331375   0.331375   0.331375   0.331375   0.331375   
       2      1          0.154821   0.154821   0.154821   0.154821   0.154821   
              2          0.106326   0.106326   0.106326   0.106326   0.106326   
       3      1          0.142417   0.142417   0.142417   0.142417   0.142417   
              2          0.097807   0.097807   0.097807   0.097807   0.097807   

LSOA2021                E01008167  E01008170  E01008172  E01008173  E01008174  \
ns_sec adults children                                                          
1      1      1          0.139290   0.139290   0.139290   0.139290   0.139290   
              2          0.095660   0.095660   0.095660   0.095660   0.095660   
       2      1          0.044693   0.044693   0.044693   0.044693   0.044693   
              2          0.030694   0.030694   0.030694   0.030694   0.030694   
       3      1          0.041112   0.041112   0.041112   0.041112   0.041112   
              2          0.028235   0.028235   0.028235   0.028235   0.028235   
2      1      1          0.189666   0.189666   0.189666   0.189666   0.189666   
              2          0.130257   0.130257   0.130257   0.130257   0.130257   
       2      1          0.060857   0.060857   0.060857   0.060857   0.060857   
              2          0.041795   0.041795   0.041795   0.041795   0.041795   
       3      1          0.055981   0.055981   0.055981   0.055981   0.055981   
              2          0.038446   0.038446   0.038446   0.038446   0.038446   
3      1      1          0.388457   0.388457   0.388457   0.388457   0.388457   
              2          0.266780   0.266780   0.266780   0.266780   0.266780   
       2      1          0.124642   0.124642   0.124642   0.124642   0.124642   
              2          0.085

In [19]:
# calculate the probabilities from the logit model
probabilities = exponential.copy()
probabilities.data = 1 / (probabilities.data.add(1))
probabilities.data

LSOA2021                E01008162  E01008163  E01008164  E01008165  E01008166  \
ns_sec adults children                                                          
1      1      1          0.877739   0.877739   0.877739   0.877739   0.877739   
              2          0.912692   0.912692   0.912692   0.912692   0.912692   
       2      1          0.957219   0.957219   0.957219   0.957219   0.957219   
              2          0.970220   0.970220   0.970220   0.970220   0.970220   
       3      1          0.960511   0.960511   0.960511   0.960511   0.960511   
              2          0.972541   0.972541   0.972541   0.972541   0.972541   
2      1      1          0.840572   0.840572   0.840572   0.840572   0.840572   
              2          0.884755   0.884755   0.884755   0.884755   0.884755   
       2      1          0.942634   0.942634   0.942634   0.942634   0.942634   
              2          0.959882   0.959882   0.959882   0.959882   0.959882   
       3      1          0.946987   0.946987   0.946987   0.946987   0.946987   
              2          0.962977   0.962977   0.962977   0.962977   0.962977   
3      1      1          0.720224   0.720224   0.720224   0.720224   0.720224   
              2          0.789403   0.789403   0.789403   0.789403   0.789403   
       2      1          0.889172   0.889172   0.889172   0.889172   0.889172   
              2          0.921150   0.921150   0.921150   0.921150   0.921150   
       3      1          0.897138   0.897138   0.897138   0.897138   0.897138   
              2          0.927006   0.927006   0.927006   0.927006   0.927006   
4      1      1          0.537276   0.537276   0.537276   0.537276   0.537276   
              2          0.628349   0.628349   0.628349   0.628349   0.628349   
       2      1          0.783490   0.783490   0.783490   0.783490   0.783490   
              2          0.840490   0.840490   0.840490   0.840490   0.840490   
       3      1          0.797321   0.797321   0.797321   0.797321   0.797321   
              2          0.851371   0.851371   0.851371   0.851371   0.851371   
5      1      1          0.674530   0.674530   0.674530   0.674530   0.674530   
              2          0.751103   0.751103   0.751103   0.751103   0.751103   
       2      1          0.865935   0.865935   0.865935   0.865935   0.865935   
              2          0.903893   0.903893   0.903893   0.903893   0.903893   
       3      1          0.875337   0.875337   0.875337   0.875337   0.875337   
              2          0.910907   0.910907   0.910907   0.910907   0.910907   

LSOA2021                E01008167  E01008170  E01008172  E01008173  E01008174  \
ns_sec adults children                                                          
1      1      1          0.877739   0.877739   0.877739   0.877739   0.877739   
              2          0.912692   0.912692   0.912692   0.912692   0.912692   
       2      1          0.957219   0.957219   0.957219   0.957219   0.957219   
              2          0.970220   0.970220   0.970220   0.970220   0.970220   
       3      1          0.960511   0.960511   0.960511   0.960511   0.960511   
              2          0.972541   0.972541   0.972541   0.972541   0.972541   
2      1      1          0.840572   0.840572   0.840572   0.840572   0.840572   
              2          0.884755   0.884755   0.884755   0.884755   0.884755   
       2      1          0.942634   0.942634   0.942634   0.942634   0.942634   
              2          0.959882   0.959882   0.959882   0.959882   0.959882   
       3      1          0.946987   0.946987   0.946987   0.946987   0.946987   
              2          0.962977   0.962977   0.962977   0.962977   0.962977   
3      1      1          0.720224   0.720224   0.720224   0.720224   0.720224   
              2          0.789403   0.789403   0.789403   0.789403   0.789403   
       2      1          0.889172   0.889172   0.889172   0.889172   0.889172   
              2          0.921

In [20]:
_1_plus = probabilities * hh_data_aggregated
_no_car = hh_data_aggregated - _1_plus

In [21]:
_1_plus.data

LSOA2021                 E01008162  E01008163  E01008164  E01008165  \
ns_sec adults children                                                
1      1      1         181.431043  71.856417  48.125004  21.385913   
              2           9.873695   6.139214   7.559140   6.995593   
       2      1         153.288203  55.940232  51.234218  24.798542   
              2          65.544311  42.799450  34.857154  22.567688   
       3      1          30.514722  23.043520  10.288968  12.067093   
              2           9.739644  20.290878   7.514177   6.068654   
2      1      1          91.663584  62.395580  57.613438  29.710595   
              2           6.459819   5.382385   7.549495  12.701729   
       2      1          83.960532  40.161531  50.342038  35.884683   
              2          29.280745  21.287412  20.128008  22.771794   
       3      1          21.993038  18.549084   9.764689  20.472282   
              2           6.597154  14.749312   6.009058   9.521629   
3      1      1          96.271260  68.235122  70.966234  43.710373   
              2           7.354856   7.930569  11.352181  25.342644   
       2      1          77.434388  48.619386  62.802824  57.970857   
              2          20.051458  21.748035  19.896386  28.948559   
       3      1          19.367336  24.670930  12.283405  33.537003   
              2           5.166011  17.078727   6.640848  13.164667   
4      1      1          33.373354  23.885350  32.417316  22.414951   
              2           1.900312   2.558180   4.174184  11.085010   
       2      1          18.857483  13.071825  22.515995  23.073152   
              2           2.842887   3.813564   4.322802   7.650515   
       3      1           2.869277   4.702865   2.872447   9.525010   
              2           0.689765   2.854276   1.308159   3.525471   
5      1      1          15.505777  10.304518   5.046718   2.456152   
              2           3.379504   2.977485   2.492677   2.977816   
       2      1           8.540596   3.707395   3.355361   1.810559   
              2           5.336328   3.089807   2.591716   2.124005   
       3      1           2.533349   1.670216   0.798511   1.191569   
              2           0.776421   1.048875   0.496558   0.539391   

LSOA2021                 E01008166  E01008167   E01008170  E01008172  \
ns_sec adults children                                                 
1      1      1         124.643674  50.208083   69.904696  60.520136   
              2           0.963070   4.575503    5.385658   7.008988   
       2      1          71.412380  53.249122   58.255680  90.406621   
              2           7.118454  33.445017   30.443229  34.980981   
       3      1           7.682642  17.419795   24.688193  22.298531   
              2           0.000000  10.259028   10.228465   9.388312   
2      1      1          48.871228  53.349103   74.574829  64.595677   
              2           0.459361   5.307955    6.019737   8.003208   
       2      1          27.537806  50.655378   48.960712  91.244089   
              2           2.509000  22.588721   18.041756  25.366735   
       3      1           4.273572  19.046139   22.344429  26.768987   
              2           0.000000   9.773961    7.844002  10.222596   
3      1      1          78.168829  55.624021  122.939430  64.469325   
              2           0.942307   6.837605   12.438811   9.889365   
       2      1          44.788255  54.046080   71.456370  77.551161   
              2           3.478972  19.403357   20.643003  15.471784   
       3      1           7.289000  21.235790   26.932298  22.085205   
              2           0.000000   8.889381    8.114590   6.793684   
4      1      1          38.758747  28.481619   49.342819  26.988573   
              2           0.479198   3.308162    4.966017   3.923664   
       2      1          19.693011  21.046033   25.358369  24.185310   
              2           1.239983   5.331555    5.306226   2.919960   

In [22]:
_no_car.data

LSOA2021                E01008162  E01008163  E01008164  E01008165  E01008166  \
ns_sec adults children                                                          
1      1      1         25.271596  10.008907   6.703349   2.978852  17.361663   
              2          0.944519   0.587278   0.723108   0.669199   0.092127   
       2      1          6.850934   2.500146   2.289819   1.108325   3.191645   
              2          2.011805   1.313678   1.069899   0.692688   0.218493   
       3      1          1.254533   0.947374   0.423004   0.496107   0.315852   
              2          0.274995   0.572906   0.212160   0.171346   0.000000   
2      1      1         17.385500  11.834344  10.927332   5.635101   9.269229   
              2          0.841435   0.701092   0.983372   1.654486   0.059835   
       2      1          5.109584   2.444109   3.063664   2.183833   1.675868   
              2          1.223777   0.889699   0.841242   0.951738   0.104863   
       3      1          1.231196   1.038399   0.546639   1.146062   0.239239   
              2          0.253634   0.567053   0.231025   0.366069   0.000000   
3      1      1         37.397264  26.506424  27.567344  16.979609  30.365244   
              2          1.962127   2.115716   3.028533   6.760906   0.251388   
       2      1          9.651549   6.060000   7.827847   7.225583   5.582482   
              2          1.716402   1.861628   1.703128   2.477992   0.297799   
       3      1          2.220570   2.828656   1.408359   3.845200   0.835723   
              2          0.406780   1.344807   0.522911   1.036607   0.000000   
4      1      1         28.742528  20.571062  27.919148  19.304693  33.380654   
              2          1.123983   1.513094   2.468917   6.556484   0.283433   
       2      1          5.211093   3.612279   6.222089   6.376055   5.441983   
              2          0.539529   0.723746   0.820390   1.451930   0.235326   
       3      1          0.729372   1.195471   0.730178   2.421263   0.706878   
              2          0.120417   0.498290   0.228374   0.615465   0.000000   
5      1      1          7.481756   4.972075   2.435113   1.185128   4.631243   
              2          1.119883   0.986664   0.826011   0.986774   0.114829   
       2      1          1.322264   0.573983   0.519481   0.280313   0.621143   
              2          0.567391   0.328527   0.275567   0.225837   0.076707   
       3      1          0.360792   0.237867   0.113721   0.169700   0.109450   
              2          0.075940   0.102588   0.048567   0.052756   0.000000   

LSOA2021                E01008167  E01008170  E01008172  E01008173  E01008174  \
ns_sec adults children                                                          
1      1      1          6.993502   9.737051   8.429872   2.559932   7.624637   
              2          0.437693   0.515193   0.670481   0.587457   0.792628   
       2      1          2.379872   2.603630   4.040558   1.165643   4.610117   
              2          1.026555   0.934419   1.073700   0.696653   2.301283   
       3      1          0.716169   1.014990   0.916746   0.614458   1.447612   
              2          0.289660   0.288797   0.265076   0.195859   0.572849   
2      1      1         10.118531  14.144338  12.251628   6.277868  10.068901   
              2          0.691397   0.784111   1.042472   1.550705   1.065530   
       2      1          3.082733   2.979601   5.552839   2.583862   4.333486   
              2          0.944087   0.754048   1.060193   1.075512   1.454222   
       3      1          1.066225   1.250867   1.498559   1.572329   1.499156   
              2          0.375770   0.301571   0.393018   0.468604   0.537909   
3      1      1         21.607551  47.756706  25.043573  20.551968  34.359468   
              2          1.824135   3.318424   2.638283   5.750772   3.845015   
       2      1          6.736392   8.906439   9.666104   7.188502   9.862273   
              2          1.660

In [ ]:
# TODO: then this would be applied in the sub model again of 1v2+